# 2.5 Verdieping: Simpson's Paradox & Subgroep-analyse (Jeffrey & Rick)

In **Notebook 02.3** hebben we gezien hoe een geaggregeerde trend kan vertekenen of zelfs compleet kan omdraaien door een onderliggende verstorende variabele (*confounder*). In **Notebook 02.4** zagen we dat het tellen van berichten vs. individuen scheefheden kan maskeren.

In dit notebook onderzoeken we twee mogelijke verstorende variabelen voor onze **Hero Visualisatie** (*Communicatiepiek in de middag!*):
1. **Periode (Weekdag vs. Weekend):** Geldt de middagpiek op alle dagen, of communiceren we in het weekend heel anders?
2. **Persoon (Partner vs. Student):** Komt de middagpiek doordat één iemand veel stuurt in de middag, of volgen beiden hetzelfde ritme?

Dit notebook dient als **empirische achtervang** voor de pitch van Opdracht 1: hiermee toon je aan dat je het effect kritisch hebt ontleed op subgroepniveau.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from wa_analyzer.data import ROOT, load_own_chat

# 1. Data inladen en filteren
df = load_own_chat()
clean = df[~df['has_only_media'] & ~df['is_deleted'] & ~df['only_emoji']].copy()
clean['ts'] = pd.to_datetime(clean['timestamp'])
clean['is_weekend'] = clean['ts'].dt.weekday.isin([5, 6])
clean['period'] = np.where(clean['is_weekend'], 'Weekend (za-zo)', 'Weekdag (ma-vr)')

day_order = ['Ochtend', 'Middag', 'Avond', 'Nacht']

print(f"Totale berichten: {len(df):,}".replace(',', '.'))
print(f"Inhoudelijke berichten: {len(clean):,} (uitgefilterd: {len(df) - len(clean)})".replace(',', '.'))
print(f"Aantal actieve dagen: {clean['ts'].dt.date.nunique()}")

## 1. Verdieping A: Weekdag vs. Weekend (% uitsplitsing)

Omdat er 5 weekdagen en slechts 2 weekenddagen zijn, vergelijken we **relatieve percentages** binnen elke periode (de noemer is het totaal aantal berichten in die periode).

* **Weekdagen (n=4.996):** 48,6% van de berichten valt in de middag. Hier zien we het typische werk/studieritme.
* **Weekend (n=1.063):** De ochtend stijgt van 32,8% naar **39,8%**, en wordt daarmee nét actiever dan de middag (38,8%). In het weekend verschuift de piek dus subtiel naar de ochtend!

In [ ]:
# Tabel: Aantallen en percentages per periode
counts_period = clean.groupby(['period', 'day_part'], observed=True).size().unstack()[day_order]
pct_period = counts_period.div(counts_period.sum(axis=1), axis=0) * 100

df_periode = pd.DataFrame({
    'Weekdag (n)': counts_period.loc['Weekdag (ma-vr)'],
    'Weekdag (%)': pct_period.loc['Weekdag (ma-vr)'].round(1),
    'Weekend (n)': counts_period.loc['Weekend (za-zo)'],
    'Weekend (%)': pct_period.loc['Weekend (za-zo)'].round(1),
})
df_periode

## 2. Verdieping B: Partner vs. Student (Barbell Plot)

In **Notebook 02.3** leerden we de **Barbell plot** (haltergrafiek) kennen als de ideale visualisatie om het verschil (*the gap*) tussen twee groepen direct zichtbaar te maken.

Hieronder berekenen we het percentage van eigen berichten per dagdeel voor Partner vs. Student. Zoals we zien is de *gap* nagenoeg nul: in de middag sturen beiden exact 46,8% vs. 46,9% van hun berichten!

In [ ]:
# Tabel: Partner vs Student percentages en gap
counts_author = clean.groupby(['author', 'day_part'], observed=True).size().unstack()[day_order]
pct_author = counts_author.div(counts_author.sum(axis=1), axis=0) * 100

gap_df = pd.DataFrame({
    'Partner (%)': pct_author.loc['Partner'].round(1),
    'Student (%)': pct_author.loc['Student'].round(1),
    'Gap (%-punt)': (pct_author.loc['Student'] - pct_author.loc['Partner']).abs().round(1),
})
gap_df

## 3. De Gecombineerde Verdiepingsvisualisatie

We combineren beide vergelijkingen in één compacte, professionele visualisatie volgens de Schwabish-principes:
* **Deel A:** Twee panelen met gedeelde Y-as voor Weekdag vs. Weekend (met WhatsApp-groene focus op de piekstaven).
* **Deel B:** Een Barbell plot voor Partner vs. Student, chronologisch geordend van Ochtend tot Nacht.
* **Onderkant:** Een bescheiden, cursieve noot met noemer en filterverantwoording.

In [ ]:
# Figuur opzetten (2 rijen: boven periode, onder barbell plot)
fig = plt.figure(figsize=(10, 8.5), dpi=150)
gs = fig.add_gridspec(2, 2, height_ratios=[1.1, 1], hspace=0.45, wspace=0.22)

# Panel 1: Weekdag
ax1 = fig.add_subplot(gs[0, 0])
wd_pct = pct_period.loc['Weekdag (ma-vr)']
wd_cnt = counts_period.loc['Weekdag (ma-vr)']
colors_wd = ['#c8c8c8', '#25D366', '#c8c8c8', '#c8c8c8']
bars1 = ax1.bar(day_order, wd_pct, color=colors_wd, width=0.55)
sns.despine(ax=ax1, top=True, right=True)
ax1.set_ylim(0, 62)
ax1.set_ylabel('Percentage van weekdagberichten (%)', fontsize=10, color='#444444')
ax1.set_title('A1. Weekdagen (ma–vr, n=4.996)\nPiek overtuigend in de middag (48,6%)', fontsize=10.5, fontweight='bold', ha='left', x=0, pad=10)
ax1.grid(axis='y', linestyle=':', alpha=0.35, color='#cccccc')
for b, p, c in zip(bars1, wd_pct, wd_cnt):
    ax1.text(b.get_x() + b.get_width()/2, p + 1.2, f'{p:.1f}%'.replace('.', ','), ha='center', va='bottom', fontsize=9.5, fontweight='bold', color='#111111' if p > 40 else '#555555')
    ax1.text(b.get_x() + b.get_width()/2, p/2, f'n={c:,}'.replace(',', '.'), ha='center', va='center', fontsize=8, color='#ffffff' if p > 40 else '#444444')

# Panel 2: Weekend
ax2 = fig.add_subplot(gs[0, 1], sharey=ax1)
we_pct = pct_period.loc['Weekend (za-zo)']
we_cnt = counts_period.loc['Weekend (za-zo)']
colors_we = ['#25D366', '#a3e4b7', '#c8c8c8', '#c8c8c8']
bars2 = ax2.bar(day_order, we_pct, color=colors_we, width=0.55)
sns.despine(ax=ax2, top=True, right=True, left=True)
ax2.tick_params(left=False, labelleft=False)
ax2.set_title('A2. Weekend (za–zo, n=1.063)\nOchtend nét actiever (39,8% vs. 38,8%)', fontsize=10.5, fontweight='bold', ha='left', x=0, pad=10)
ax2.grid(axis='y', linestyle=':', alpha=0.35, color='#cccccc')
for b, p, c in zip(bars2, we_pct, we_cnt):
    ax2.text(b.get_x() + b.get_width()/2, p + 1.2, f'{p:.1f}%'.replace('.', ','), ha='center', va='bottom', fontsize=9.5, fontweight='bold', color='#111111' if p > 35 else '#555555')
    ax2.text(b.get_x() + b.get_width()/2, p/2, f'n={c:,}'.replace(',', '.'), ha='center', va='center', fontsize=8, color='#ffffff' if p > 35 else '#444444')

# Panel 3: Barbell plot Partner vs Student (chronologisch)
ax3 = fig.add_subplot(gs[1, :])
day_order_reversed = ['Nacht', 'Avond', 'Middag', 'Ochtend']
y_pos = np.arange(len(day_order_reversed))
partner_p = pct_author.loc['Partner', day_order_reversed].to_numpy()
student_p = pct_author.loc['Student', day_order_reversed].to_numpy()

for y, p1, p2 in zip(y_pos, partner_p, student_p):
    ax3.hlines(y=y, xmin=min(p1, p2), xmax=max(p1, p2), color='#888888', linewidth=3.5, zorder=1)

ax3.scatter(partner_p, y_pos, s=120, color='#128C7E', label='Partner (n=2.783)', zorder=3)
ax3.scatter(student_p, y_pos, s=120, color='#25D366', label='Student / Jeffrey (n=3.276)', zorder=3)

sns.despine(ax=ax3, top=True, right=True)
ax3.set_yticks(y_pos)
ax3.set_yticklabels(day_order_reversed, fontsize=11)
ax3.set_xlim(-2, 75)
ax3.set_xlabel('Percentage van eigen berichten (%)', fontsize=10, color='#444444')
ax3.set_title("B. Vergelijking per persoon (Barbell plot) — Geen Simpson's paradox: beide profielen zijn vrijwel identiek", fontsize=11, fontweight='bold', ha='left', x=0, pad=10)
ax3.grid(axis='x', linestyle=':', alpha=0.35, color='#cccccc')
ax3.legend(loc='lower right', bbox_to_anchor=(0.98, 0.05), frameon=True, facecolor='#ffffff', edgecolor='#dddddd')

for y, p1, p2, day in zip(y_pos, partner_p, student_p, day_order_reversed):
    gap = abs(p1 - p2)
    max_p = max(p1, p2)
    text_label = f'Partner {p1:.1f}%  |  Student {p2:.1f}%  (gap: {gap:.1f}%)'.replace('.', ',')
    ax3.text(max_p + 1.8, y, text_label, va='center', fontsize=9.5, color='#333333')

# Hoofdtitel & Footnote
fig.suptitle("Verdieping Dagritme & Simpson's Paradox: Periode vs. Persoon", fontsize=13.5, fontweight='bold', x=0.07, y=0.98, ha='left')

n_total = len(df)
n_clean = len(clean)
n_filtered = n_total - n_clean
n_days = clean['ts'].dt.date.nunique()
min_year = clean['ts'].dt.year.min()
max_year = clean['ts'].dt.year.max()

fig.text(0.07, 0.02, f'Noot: Gebaseerd op {n_clean:,} inhoudelijke berichten ({n_filtered} media-, verwijderde en pure emoji-berichten gefilterd) over {n_days} actieve dagen ({min_year}–{max_year}).'.replace(',', '.'), fontsize=8, style='italic', color='#777777')

out_file = ROOT / 'img' / 'dagritme_verdieping_simpsons.png'
out_file.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_file, bbox_inches='tight')
plt.show()
print(f'Opgeslagen als {out_file}')

## 4. Reflectie voor de Pitch (Achtervang)

Als docent Raoul of een medestudent vraagt:
> *"Klopt die middagpiek eigenlijk wel voor beide personen? En communiceren jullie in het weekend niet heel anders?"*

Dan heb je direct het perfecte antwoord klaar:
1. **Tussen personen is er geen Simpson's paradox:** De barbell plot toont aan dat de communicatieprofielen van Jeffrey en Rick nagenoeg identiek zijn (beiden ~47% in de middag, gap ≤ 2%). De piek is dus geen artefact van één veelprater.
2. **Tussen week en weekend is er wél een interessante verschuiving:** Op weekdagen piekt de chat zwaar in de middag (48,6%), maar in het weekend verschuift de piek licht naar de ochtend (39,8% vs. 38,8%). Omdat 82% van alle berichten op weekdagen valt, domineert de middagpiek het geaggregeerde beeld.